In [1]:
! pip install pdfplumber pandas openai tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 57.9 MB/s eta 0:00:00


In [2]:
!pip install -U langchain-text-splitters langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 30.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


#RAG용 문서 전처리

In [ ]:
import os
import re
import json
import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 텍스트 추출 함수
def extract_text_from_pdf(pdf_path):
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            try:
                text = page.extract_text()
                if text:
                    full_text += f"\n--- page{i+1} ---\n"
                    full_text += text
            except Exception as e:
                print(f"{pdf_path} {i+1}페이지 에러:", e)
    return full_text

# 2. 구조화 및 청킹 함수
def refine_and_chunk_data(full_text, univ_name):
    structured_data = []
    lines = full_text.split('\n')

    # RAG 최적화를 위한 청킹 설정
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100,
        separators=["\n\n", "\n", ". ", " "]
    )

    # 제어 문자 제거를 위한 정규식 (엑셀 오류 방지용이지만 JSON 데이터 정제에도 좋음)
    ILLEGAL_CHARACTERS_RE = re.compile(r'[\000-\010]|[\013-\014]|[\016-\037]')

    adm_pattern = re.compile(r"^[ⅠⅡⅢⅣⅤ]+[-.\d]*\s*([가-힣\s\(\)]+전형.*)")
    sec_pattern = re.compile(r"^([1-5])\.?\s*(지원자격|전형방법|제출서류|수능\s?최저)")

    current_adm = "공통/미분류"
    current_sec = "기본정보"
    temp_content = []

    def add_chunks(content_list, adm, sec):
        full_paragraph = " ".join(content_list)
        # 불법 제어 문자 정제
        full_paragraph = ILLEGAL_CHARACTERS_RE.sub("", full_paragraph)

        chunks = text_splitter.split_text(full_paragraph)
        for i, chunk in enumerate(chunks):
            structured_data.append({
                "university": univ_name,
                "admission_type": adm,
                "section": sec,
                "content": chunk,
                "chunk_id": i + 1
            })

    for line in lines:
        line = line.strip()
        if "····" in line or re.match(r"^---\s*page\d+\s*---$", line) or not line:
            continue

        adm_match = adm_pattern.match(line)
        if adm_match:
            if temp_content:
                add_chunks(temp_content, current_adm, current_sec)
            current_adm = adm_match.group(1).strip()
            current_sec = "전형개요"
            temp_content = []
            continue

        sec_match = sec_pattern.match(line)
        if sec_match:
            if temp_content:
                add_chunks(temp_content, current_adm, current_sec)
            current_sec = sec_match.group(2).strip()
            temp_content = []
            continue

        temp_content.append(line)

    if temp_content:
        add_chunks(temp_content, current_adm, current_sec)

    return structured_data

# 3. 메인 실행부
pdf_folder = './pdfs'
all_university_data = []

if not os.path.exists(pdf_folder):
    os.makedirs(pdf_folder)
    print(f"'{pdf_folder}' 폴더에 PDF를 넣어주세요.")

for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):
        # 파일명에서 대학명 추출 (파일명 예: snu_2026.pdf)
        univ_name = filename.split('_')[0]
        file_path = os.path.join(pdf_folder, filename)

        print(f"[{univ_name}] 처리 중...")
        raw_text = extract_text_from_pdf(file_path)
        refined_list = refine_and_chunk_data(raw_text, univ_name)
        all_university_data.extend(refined_list)

# 4. 통합 JSON 저장
output_path = 'integrated_admission_data.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_university_data, f, ensure_ascii=False, indent=4)

print(f"\n✅ {len(all_university_data)}개의 청크가 '{output_path}'에 저장되었습니다.")

[연세대학교] 처리 중...
[시립대학교] 처리 중...
[한양대학교] 처리 중...
[연세대학교] 처리 중...
[강원대학교] 처리 중...
[고려대학교] 처리 중...
[한양대학교] 처리 중...
[고려대학교] 처리 중...
[강원대학교] 처리 중...
[서울대학교] 처리 중...
[서울대학교] 처리 중...
[시립대학교] 처리 중...
[한림대학교] 처리 중...
[한림대학교] 처리 중...

✅ 2771개의 청크가 'integrated_admission_data.json'에 저장되었습니다.


In [ ]:
import os
import re
import json
import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 텍스트 추출 함수
def extract_text_from_pdf(pdf_path):
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            try:
                text = page.extract_text()
                if text:
                    full_text += f"\n--- page{i+1} ---\n"
                    full_text += text
            except Exception as e:
                print(f"{pdf_path} {i+1}페이지 에러:", e)
    return full_text

# 2. 구조화 및 청킹 함수 (season 인자 추가)
def refine_and_chunk_data(full_text, univ_name, season):
    structured_data = []
    lines = full_text.split('\n')
    # RAG 최적화를 위한 청킹 설정
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100,
        separators=["\n\n", "\n", ". ", " "]
    )
    # 제어 문자 제거를 위한 정규식 (엑셀 오류 방지용이지만 JSON 데이터 정제에도 좋음)
    ILLEGAL_CHARACTERS_RE = re.compile(r'[\000-\010]|[\013-\014]|[\016-\037]')
    adm_pattern = re.compile(r"^[ⅠⅡⅢⅣⅤ]+[-.\d]*\s*([가-힣\s\(\)]+전형.*)")
    sec_pattern = re.compile(r"^([1-5])\.?\s*(지원자격|전형방법|제출서류|수능\s?최저)")

    current_adm = "공통/미분류"
    current_sec = "기본정보"
    temp_content = []

    def add_chunks(content_list, adm, sec):
        full_paragraph = " ".join(content_list)
        full_paragraph = ILLEGAL_CHARACTERS_RE.sub("", full_paragraph)

        chunks = text_splitter.split_text(full_paragraph)
        for i, chunk in enumerate(chunks):
            structured_data.append({
                "university": univ_name,
                "admission_season": season,  # 수시/정시 구분 추가
                "admission_type": adm,
                "section": sec,
                "content": chunk,
                "chunk_id": i + 1
            })

    for line in lines:
        line = line.strip()
        if "····" in line or re.match(r"^---\s*page\d+\s*---$", line) or not line:
            continue

        adm_match = adm_pattern.match(line)
        if adm_match:
            if temp_content:
                add_chunks(temp_content, current_adm, current_sec)
            current_adm = adm_match.group(1).strip()
            current_sec = "전형개요"
            temp_content = []
            continue

        sec_match = sec_pattern.match(line)
        if sec_match:
            if temp_content:
                add_chunks(temp_content, current_adm, current_sec)
            current_sec = sec_match.group(2).strip()
            temp_content = []
            continue

        temp_content.append(line)

    if temp_content:
        add_chunks(temp_content, current_adm, current_sec)

    return structured_data

# 3. 메인 실행부
pdf_folder = './pdfs'
all_university_data = []

for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):
        # 파일명 분석 (예: snu_susi.pdf -> univ_name: snu, season: 수시)
        # '_'를 기준으로 분리
        parts = filename.replace('.pdf', '').split('_')
        univ_name = parts[0]

        # 파일명에 'susi'나 'jungsi'가 포함되어 있는지 확인하여 한글로 변환
        raw_season = parts[1].lower() if len(parts) > 1 else "미분류"
        season = "수시" if "susi" in raw_season else "정시" if "jungsi" in raw_season else raw_season

        file_path = os.path.join(pdf_folder, filename)

        print(f"[{univ_name} | {season}] 처리 중...")
        raw_text = extract_text_from_pdf(file_path)
        refined_list = refine_and_chunk_data(raw_text, univ_name, season)
        all_university_data.extend(refined_list)

# 4. 통합 JSON 저장
output_path = 'integrated_admission_data.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_university_data, f, ensure_ascii=False, indent=4)

print(f"\n✅ 수시/정시 구분 포함 {len(all_university_data)}개 데이터 저장 완료!")

[연세대학교 | 2026 수시] 처리 중...
[시립대학교 | 2026 수시] 처리 중...
[한양대학교 | 2026 수시] 처리 중...
[연세대학교 | 2026 정시] 처리 중...
[강원대학교 | 2026 정시] 처리 중...
[고려대학교 | 2026 수시] 처리 중...
[한양대학교 | 2026 정시] 처리 중...
[고려대학교 | 2026 정시] 처리 중...
[강원대학교 | 2026] 처리 중...
[서울대학교 | 2026 정시] 처리 중...
[서울대학교 | 2026 수시] 처리 중...
[시립대학교 | 2026.정시pdf] 처리 중...
[한림대학교 | 2026 정시] 처리 중...
[한림대학교 | 2026 수시] 처리 중...

✅ 수시/정시 구분 포함 2771개 데이터 저장 완료!


In [ ]:
import pandas as pd
import json

# 1. JSON 파일 불러오기
with open('integrated_admission_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. 데이터프레임으로 변환
df = pd.DataFrame(data)

# 3. 엑셀 파일로 저장 (검수하기 좋게 컬럼 순서 조정)
column_order = ['university', 'admission_season', 'admission_type', 'section', 'chunk_id', 'content']
df = df[column_order]

df.to_excel('final_check_data.xlsx', index=False)

print(f"✅ 검수용 엑셀 파일 'final_check_data.xlsx'가 생성되었습니다.")

✅ 검수용 엑셀 파일 'final_check_data.xlsx'가 생성되었습니다.


# 검색 정확도 높이기_아직 성능 낮음


In [3]:
import os
import re
import json
import pdfplumber
from collections import Counter

# 1. PDF 텍스트 추출
def extract_text_from_pdf(pdf_path):
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            try:
                text = page.extract_text()
                if text:
                    full_text += f"\n--- page{i+1} ---\n"
                    full_text += text
            except Exception as e:
                print(f"{pdf_path} {i+1}페이지 에러:", e)
    return full_text



# 2. 키워드 추출
def extract_keywords(text, top_k=5):
    words = re.findall(r"[가-힣A-Za-z]{2,}", text)
    stopwords = ["전형", "지원", "학생", "경우", "사항", "해당", "수시", "정시"]

    words = [w for w in words if w not in stopwords]
    freq = Counter(words)

    return [w for w, _ in freq.most_common(top_k)]



# 3. 구조화 + 의미 기반 Chunking
def refine_and_chunk_data(full_text, univ_name, season):
    structured_data = []
    lines = full_text.split('\n')

    ILLEGAL_CHARACTERS_RE = re.compile(r'[\000-\010]|[\013-\014]|[\016-\037]')

    # 전형 탐지
    adm_pattern = re.compile(r"^[ⅠⅡⅢⅣⅤ]+[-.\d]*\s*([가-힣\s\(\)]+전형.*)")

    # 섹션 탐지 (확장 가능)
    sec_pattern = re.compile(
        r"(지원자격|전형방법|제출서류|수능\s?최저|전형일정|평가방법|유의사항)"
    )

    current_adm = "공통"
    current_sec = "기타"
    current_page = 0

    temp_chunk = []
    chunk_counter = 0

    def save_chunk(chunk_lines):
        nonlocal chunk_counter

        if not chunk_lines:
            return

        text = " ".join(chunk_lines)
        text = ILLEGAL_CHARACTERS_RE.sub("", text)

        # 너무 짧은 chunk 제거
        if len(text) < 50:
            return

        chunk_counter += 1

        structured_data.append({
            "chunk_id": f"{univ_name}_{chunk_counter}",
            "university": univ_name,
            "admission_season": season,
            "admission_type": "수시" if season == "수시" else "정시",
            "admission_detail_type": current_adm,
            "section": current_sec,
            "page": current_page,
            "keywords": extract_keywords(text),
            "content": text
        })

    for line in lines:
        line = line.strip()

        # page 추적
        page_match = re.match(r"--- page(\d+) ---", line)
        if page_match:
            current_page = int(page_match.group(1))
            continue

        if not line or "····" in line:
            continue

        # 전형 변경
        adm_match = adm_pattern.match(line)
        if adm_match:
            save_chunk(temp_chunk)
            current_adm = adm_match.group(1).strip()
            current_sec = "전형개요"
            temp_chunk = []
            continue

        # 섹션 변경
        sec_match = sec_pattern.search(line)
        if sec_match:
            save_chunk(temp_chunk)
            current_sec = sec_match.group(1)
            temp_chunk = []
            continue

        # bullet 기준 chunk 분리
        if line.startswith(("•", "-", "※")):
            save_chunk(temp_chunk)
            temp_chunk = [line]
            continue

        temp_chunk.append(line)

    # 마지막 chunk 저장
    save_chunk(temp_chunk)

    return structured_data



# 4. 메인 실행
pdf_folder = './pdfs'
all_university_data = []

for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):

        # 파일명 기반 parsing
        parts = filename.replace('.pdf', '').split('_')
        univ_name = parts[0]

        raw_season = parts[1].lower() if len(parts) > 1 else "미분류"
        season = "수시" if "susi" in raw_season else "정시" if "jungsi" in raw_season else raw_season

        file_path = os.path.join(pdf_folder, filename)

        print(f"[{univ_name} | {season}] 처리 중...")

        raw_text = extract_text_from_pdf(file_path)
        refined_list = refine_and_chunk_data(raw_text, univ_name, season)

        all_university_data.extend(refined_list)



# 5. JSON 저장
output_path = 'integrated_admission_data_v2.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_university_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ 총 {len(all_university_data)}개 chunk 저장 완료!")



# 6. (선택) Excel 저장
try:
    import pandas as pd

    df = pd.DataFrame(all_university_data)
    df.to_excel("rag_ready_data.xlsx", index=False)

    print("📊 Excel 파일 저장 완료!")
except:
    print("pandas 없으면 Excel 저장 생략됨")


## 관련 chunk 잘 가져옴. retrieval 자체는 되나 성능이 낮음  => semantic search에서 전형별로 필터링 실패, 정확한 QA 불가능, hallucination 위험 가능성 높다.

[연세대학교 | 2026 수시] 처리 중...
[고려대학교 | 2026 정시] 처리 중...
[시립대학교 | 2026.정시pdf] 처리 중...
[강원대학교 | 2026] 처리 중...
[서울대학교 | 2026 정시] 처리 중...
[한림대학교 | 2026 수시] 처리 중...
[한림대학교 | 2026 정시] 처리 중...
[고려대학교 | 2026 수시] 처리 중...
[한양대학교 | 2026 수시] 처리 중...
[연세대학교 | 2026 정시] 처리 중...
[시립대학교 | 2026 수시] 처리 중...
[서울대학교 | 2026 수시] 처리 중...
[강원대학교 | 2026 정시] 처리 중...
[한양대학교 | 2026 정시] 처리 중...

✅ 총 3998개 chunk 저장 완료!
📊 Excel 파일 저장 완료!


#전형 이름 정규화

In [5]:
import os
import re
import json
import pdfplumber
from collections import Counter

# =========================
# 1. PDF 텍스트 추출
# =========================
def extract_text_from_pdf(pdf_path):
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            try:
                text = page.extract_text()
                if text:
                    full_text += f"\n--- page{i+1} ---\n"
                    full_text += text
            except Exception as e:
                print(f"{pdf_path} {i+1}페이지 에러:", e)
    return full_text


# =========================
# 2. 키워드 추출
# =========================
def extract_keywords(text, top_k=5):
    words = re.findall(r"[가-힣A-Za-z]{2,}", text)

    stopwords = ["전형", "지원", "학생", "경우", "사항", "해당", "수시", "정시"]

    words = [w for w in words if w not in stopwords]
    freq = Counter(words)

    return [w for w, _ in freq.most_common(top_k)]


# =========================
# 3. 전형 정규화 (🔥 핵심)
# =========================
def normalize_admission_type(text):
    if "논술" in text:
        return "논술"
    elif "학생부종합" in text:
        return "학종"
    elif "교과" in text:
        return "교과"
    elif "정시" in text:
        return "정시"
    else:
        return "기타"


# =========================
# 4. 텍스트 정제 (🔥 중요)
# =========================
def clean_text(text,univ_name):
    # 페이지 관련 제거
    text = re.sub(r"\d+\s*페이지", "", text)

    # 대학명 반복 제거 (필요 시 확장)
    text = re.sub(fr"{univ_name}.*?캠퍼스", "", text)

    # 안내문 제거
    text = re.sub(r"※.*?참조", "", text)

    # 특수문자/공백 정리
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# =========================
# 5. 구조화 + Chunking
# =========================
def refine_and_chunk_data(full_text, univ_name, season):
    structured_data = []
    lines = full_text.split('\n')

    ILLEGAL_CHARACTERS_RE = re.compile(r'[\000-\010]|[\013-\014]|[\016-\037]')

    # 전형 탐지
    adm_pattern = re.compile(r"^[ⅠⅡⅢⅣⅤ]+[-.\d]*\s*([가-힣\s\(\)]+전형.*)")

    # 섹션 탐지
    sec_pattern = re.compile(
        r"(지원자격|전형방법|제출서류|수능\s?최저|전형일정|평가방법|유의사항)"
    )

    current_adm = "공통"
    current_sec = "기타"
    current_page = 0

    temp_chunk = []
    chunk_counter = 0

    MAX_LEN = 800  # 🔥 chunk 길이 제한

    def save_chunk(chunk_lines):
        nonlocal chunk_counter

        if not chunk_lines:
            return

        text = " ".join(chunk_lines)
        text = clean_text(text, univ_name)
        text = ILLEGAL_CHARACTERS_RE.sub("", text)

        if len(text) < 50:
            return

        # 🔥 긴 chunk 분할
        sub_chunks = [text[i:i+MAX_LEN] for i in range(0, len(text), MAX_LEN)]

        for sub_text in sub_chunks:
            chunk_counter += 1

            structured_data.append({
                "chunk_id": f"{univ_name}_{chunk_counter}",
                "university": univ_name,
                "admission_season": season,
                "admission_type": "수시" if season == "수시" else "정시",
                "admission_detail_type": current_adm,
                "section": current_sec,
                "page": current_page,

                # 🔥 강화된 metadata
                "has_nonsul": 1 if "논술" in sub_text else 0,
                "has_suneung": 1 if "수능" in sub_text else 0,
                "text_length": len(sub_text),

                "keywords": extract_keywords(sub_text),
                "content": sub_text
            })

    for line in lines:
        line = line.strip()

        # 페이지 추적
        page_match = re.match(r"--- page(\d+) ---", line)
        if page_match:
            current_page = int(page_match.group(1))
            continue

        if not line or "····" in line:
            continue

        # 🔥 전형 변경
        adm_match = adm_pattern.match(line)
        if adm_match:
            save_chunk(temp_chunk)

            raw_adm = adm_match.group(1).strip()
            current_adm = normalize_admission_type(raw_adm)

            current_sec = "전형개요"
            temp_chunk = []
            continue

        # 🔥 섹션 변경
        sec_match = sec_pattern.search(line)
        if sec_match:
            save_chunk(temp_chunk)

            current_sec = sec_match.group(1)
            temp_chunk = []
            continue

        # ❌ bullet 기준 분리 제거 (중요)
        temp_chunk.append(line)

    # 마지막 chunk 저장
    save_chunk(temp_chunk)

    return structured_data


# =========================
# 6. 메인 실행
# =========================
pdf_folder = './pdfs'
all_university_data = []

for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):

        parts = filename.replace('.pdf', '').split('_')
        univ_name = parts[0]

        raw_season = parts[1].lower() if len(parts) > 1 else "미분류"
        season = "수시" if "susi" in raw_season else "정시" if "jungsi" in raw_season else raw_season

        file_path = os.path.join(pdf_folder, filename)

        print(f"[{univ_name} | {season}] 처리 중...")

        raw_text = extract_text_from_pdf(file_path)
        refined_list = refine_and_chunk_data(raw_text, univ_name, season)

        all_university_data.extend(refined_list)


# =========================
# 7. JSON 저장
# =========================
output_path = 'integrated_admission_data_v3.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(all_university_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ 총 {len(all_university_data)}개 chunk 저장 완료!")


# =========================
# 8. Excel 저장
# =========================
try:
    import pandas as pd

    df = pd.DataFrame(all_university_data)
    df.to_excel("rag_ready_data_v3.xlsx", index=False)

    print("📊 Excel 파일 저장 완료!")
except:
    print("pandas 없으면 Excel 저장 생략됨")

[연세대학교 | 2026 수시] 처리 중...
[고려대학교 | 2026 정시] 처리 중...
[시립대학교 | 2026.정시pdf] 처리 중...
[강원대학교 | 2026] 처리 중...
[서울대학교 | 2026 정시] 처리 중...
[한림대학교 | 2026 수시] 처리 중...
[한림대학교 | 2026 정시] 처리 중...
[고려대학교 | 2026 수시] 처리 중...
[한양대학교 | 2026 수시] 처리 중...
[연세대학교 | 2026 정시] 처리 중...
[시립대학교 | 2026 수시] 처리 중...
[서울대학교 | 2026 수시] 처리 중...
[강원대학교 | 2026 정시] 처리 중...
[한양대학교 | 2026 정시] 처리 중...

✅ 총 2230개 chunk 저장 완료!
📊 Excel 파일 저장 완료!


In [8]:
import os
import re
import json
import pdfplumber
from collections import Counter

# =========================
# 1. PDF 텍스트 추출
# =========================
def extract_text_from_pdf(pdf_path):
    full_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            try:
                text = page.extract_text()
                if text:
                    full_text += f"\n--- page{i+1} ---\n"
                    full_text += text
            except Exception as e:
                print(f"{pdf_path} {i+1}페이지 에러:", e)
    return full_text


# =========================
# 2. 키워드 추출
# =========================
def extract_keywords(text, top_k=5):
    words = re.findall(r"[가-힣A-Za-z]{2,}", text)

    stopwords = ["전형", "지원", "학생", "경우", "사항", "해당", "수시", "정시"]

    words = [w for w in words if w not in stopwords]
    freq = Counter(words)

    return [w for w, _ in freq.most_common(top_k)]


# =========================
# 3. 전형 정규화
# =========================
def normalize_admission_type(text):
    text = text.replace(" ", "")

    if "논술" in text:
        return "논술"
    elif "학생부종합" in text:
        return "학종"
    elif "학생부교과" in text or "교과" in text:
        return "교과"
    elif "특기자" in text:
        return "특기자"
    elif "정시" in text:
        return "정시"
    else:
        return "기타"


# =========================
# 4. 텍스트 정제
# =========================
def clean_text(text, univ_name):
    text = re.sub(fr"{univ_name}.*?캠퍼스", "", text)
    text = re.sub(r"\d+\s*페이지", "", text)
    text = re.sub(r"※.*?참조", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# =========================
# 5. 전형 판단 필터
# =========================
def is_valid_admission(line):
    return ("전형" in line) and len(line) < 50


# =========================
# 6. 구조화 + Chunking
# =========================
def refine_and_chunk_data(full_text, univ_name, season):
    structured_data = []
    lines = full_text.split('\n')

    ILLEGAL_CHARACTERS_RE = re.compile(r'[\000-\010]|[\013-\014]|[\016-\037]')

    # 🔥 더 유연한 전형 패턴
    adm_pattern = re.compile(r"([가-힣A-Za-z\s\(\)]+전형)")

    # 섹션 패턴
    sec_pattern = re.compile(
        r"(지원자격|전형방법|제출서류|수능\s?최저|전형일정|평가방법|유의사항)"
    )

    current_adm = "공통"
    current_sec = "기타"
    current_page = 0

    temp_chunk = []
    chunk_counter = 0

    MAX_LEN = 800

    def save_chunk(chunk_lines):
        nonlocal chunk_counter

        if not chunk_lines:
            return

        text = " ".join(chunk_lines)
        text = clean_text(text, univ_name)
        text = ILLEGAL_CHARACTERS_RE.sub("", text)

        if len(text) < 50:
            return

        sub_chunks = [text[i:i+MAX_LEN] for i in range(0, len(text), MAX_LEN)]

        for sub_text in sub_chunks:
            chunk_counter += 1

            structured_data.append({
                "chunk_id": f"{univ_name}_{chunk_counter}",
                "university": univ_name,
                "admission_season": season,
                "admission_type": season,
                "admission_detail_type": current_adm,
                "section": current_sec,
                "page": current_page,

                "has_nonsul": 1 if "논술" in sub_text else 0,
                "has_suneung": 1 if "수능" in sub_text else 0,
                "text_length": len(sub_text),

                "keywords": extract_keywords(sub_text),
                "content": sub_text
            })

    for line in lines:
        line = line.strip()

        # 페이지 추적
        page_match = re.match(r"--- page(\d+) ---", line)
        if page_match:
            current_page = int(page_match.group(1))
            continue

        if not line or "····" in line:
            continue

        # 🔥 전형 변경 감지
        adm_match = adm_pattern.search(line)
        if adm_match and is_valid_admission(line):
            save_chunk(temp_chunk)

            raw_adm = adm_match.group(1).strip()
            current_adm = normalize_admission_type(raw_adm)

            current_sec = "전형개요"
            temp_chunk = []
            continue

        # 🔥 섹션 변경
        sec_match = sec_pattern.search(line)
        if sec_match:
            save_chunk(temp_chunk)

            current_sec = sec_match.group(1)
            temp_chunk = []
            continue

        temp_chunk.append(line)

    save_chunk(temp_chunk)

    return structured_data


# # =========================
# # 7. 파일명 기반 시즌 파싱 (🔥 수정 핵심)
# # =========================
# def parse_filename(file_name):
#     name = file_name.replace(".pdf", "")
#     name = name.replace("_", " ")

#     parts = name.split()
#     univ_name = parts[0]

#     if "수시" in name:
#         season = "수시"
#     elif "정시" in name:
#         season = "정시"
#     else:
#         season = "미분류"

#     return univ_name, season


# =========================
# 8. 메인 실행
# =========================
pdf_folder = './pdfs'
all_university_data = []

for filename in os.listdir(pdf_folder):
    if filename.endswith(".pdf"):

        parts = filename.replace('.pdf', '').split('_')
        univ_name = parts[0]

        raw_season = parts[1].lower() if len(parts) > 1 else "미분류"
        season = "수시" if "susi" in raw_season else "정시" if "jungsi" in raw_season else raw_season

        file_path = os.path.join(pdf_folder, filename)

        print(f"[{univ_name} | {season}] 처리 중...")

        raw_text = extract_text_from_pdf(file_path)
        refined_list = refine_and_chunk_data(raw_text, univ_name, season)

        all_university_data.extend(refined_list)


# =========================
# 9. JSON 저장
# =========================
with open('integrated_admission_data_v4.json', 'w', encoding='utf-8') as f:
    json.dump(all_university_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ 총 {len(all_university_data)}개 chunk 저장 완료!")


# =========================
# 10. Excel 저장
# =========================
try:
    import pandas as pd

    df = pd.DataFrame(all_university_data)
    df.to_excel("rag_ready_data_v4.xlsx", index=False)

    print("📊 Excel 저장 완료!")

    # 🔥 구조 확인 (중요)
    print("\n📌 샘플 구조 확인")
    print(df[['university', 'admission_type', 'admission_detail_type', 'section']].head(20))

except:
    print("pandas 없으면 Excel 저장 생략됨")

[연세대학교 | 2026 수시] 처리 중...
[고려대학교 | 2026 정시] 처리 중...
[시립대학교 | 2026.정시pdf] 처리 중...
[강원대학교 | 2026] 처리 중...
[서울대학교 | 2026 정시] 처리 중...
[한림대학교 | 2026 수시] 처리 중...
[한림대학교 | 2026 정시] 처리 중...
[고려대학교 | 2026 수시] 처리 중...
[한양대학교 | 2026 수시] 처리 중...
[연세대학교 | 2026 정시] 처리 중...
[시립대학교 | 2026 수시] 처리 중...
[서울대학교 | 2026 수시] 처리 중...
[강원대학교 | 2026 정시] 처리 중...
[한양대학교 | 2026 정시] 처리 중...

✅ 총 2749개 chunk 저장 완료!
📊 Excel 저장 완료!

📌 샘플 구조 확인
      university admission_type admission_detail_type section
0   연세대학교      2026 수시                    기타    전형개요
1   연세대학교      2026 수시                    논술    지원자격
2   연세대학교      2026 수시                   특기자    전형개요
3   연세대학교      2026 수시                    논술    전형개요
4   연세대학교      2026 수시                    기타    전형개요
5   연세대학교      2026 수시                    기타    전형개요
6   연세대학교